In [ ]:
#################TOOLS

In [ ]:
from langchain_core.tools import tool #decoratore langchain per definire un tool utilizzabile da un agente
import requests  #libreria per fare richieste HTTPs

In [ ]:
@tool
def get_weather(location:str) -> str:  #return una string
    """Get current weather for a location.
    Use for queries about weather, temperature, or conditions in any city.
    Examples: "weather in Paris", "temperature in Tokyo", "is it raining in London"
    Args:
        location: City name (e.g., "New York", "London", "Tokyo")
    Returns:
        Current weather information including temperature and conditions.
    """
    #docstring che visto che c'è @, allora viene letta anche dall'agente
    url = f"https://wttr.in/{location}?format=j1"  #call api esterna weather api pubblica
    response = requests.get(url, timeout=10)  #chiamata http
    response.raise_for_status()  #se http error (404,500,ect) lancia eccezione
    data = response.json()   #converte risposta api in json
    return data

@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression.
    USE THIS TOOL FOR:
    - Any mathematical calculations or arithmetic operations
    - Queries involving numbers and operators (+, -, *, /, **, %)
    - Questions asking to compute, calculate, or solve math problems
    - Evaluating mathematical expressions
    EXAMPLE QUERIES:
    - "What is 2 + 2?"
    - "Calculate 15 times 7"
    - "Solve 100 / 4"
    - "What's 5 to the power of 3?"
    - "Compute 45 * 12 + 30"
    DO NOT USE FOR:
    - Word problems without explicit expressions (extract the math first)
    - Questions about mathematical concepts or theory
    Args:
        expression: Math expression like "2 + 2" or "15 * 7" (use standard Python operators)
    """
    #docstring che visto che c'è @, allora viene letta anche dall'agente
    try:
        result = eval(expression)  #eval() esegue stringa come codice python, quindi "2 + 2" -> 4
        print(f"[TOOL] calculate ('{expression}') -> '{result}'")
    except Exception as e:
        print(f"Exception has occured with error: {e}")
        return f"Exception has occured with error: {e}"
    return result

In [ ]:
#################LANGGRAPH & ReAct

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict, Annotated  #TypedDict per schema dello stato, Annotated per aggiungere metadati ai campi!
import operator   #libreria con funzioni per operatori matematici standard
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

from langgraph.prebuilt import ToolNode  #nodo prebuilt LangGraph per tool execution

# Configuration
BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3" # llama 3.2
# gemma3, deepseek-r1 -< they don't have tools support yet

llm = ChatOllama(
    model=MODEL_NAME, 
    base_url=BASE_URL)

In [ ]:
class AgentState(TypedDict):  #normale classe per stato
    messages: Annotated[list, operator.add]
    #grazie ad Annotated, puoi concatenare! FONDAMENTALE
    #e.g
    #{  
    #   "messages": [msg1]  #stato old in input
    # }
    #{  
    #   "messages": [msg1]  #stato updated
    # }
    #e ottieni merge grazie a operator.add
    # {
    #   "messages": [msg1, msg2]
    # }
    #quindi in questo modo ottieni la history

In [ ]:
import my_tools  #in realta è il code sopra sopra in alto ad inizio di questo file

# my_tools.get_weather.invoke({'location': "Mumbai"})

my_tools.calculate.invoke({'expression': '2+2*1.4/23-34'})  #esegui tool manualmente, passandogli argomenti richiesti
all_tools = [my_tools.get_weather, my_tools.calculate]  #lista tools disponibili per agente


In [ ]:
def agent_node(state: AgentState):  #nodo agente
    llm_with_tools = llm.bind_tools(all_tools)  #crea versione del llm (nuova istanza) che sa usare i tools, passandogli lista tools disponibili
    messages = state['messages']  #prendi cronologia chat
    response = llm_with_tools.invoke(messages)
    if hasattr(response, 'tool_calls') and response.tool_calls:  #se risposta possiede tool_calls (campo speciale di AIMessage) e se tool_calls è positivo, allora stampa quali tool sono stati chiamati e con quali argomenti
        for tc in response.tool_calls:
            print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
            #tc è qualcosa come
            #tc = {
            #     "name": "calculate",
            #     "args": {
            #         "expression": "2+2"
            #     }
            # }
    else:
        print(f"[AGENT] Responding...")
    return {'messages': [response]}  #update stato shared

state = {"messages": [HumanMessage("Hi")]}  #stato iniziale con una semplice domanda, poi passo questo stato al nodo agente, che lo processa e risponde, aggiornando lo stato con la nuova risposta (grazie ad operator.add in AgentState)
result = agent_node(state)  #esgue tutto il processo di ReAct, quindi se il modello decide di chiamare un tool, allora viene chiamato e la risposta del tool viene aggiunta alla history, e poi il modello risponde alla fine con la risposta finale all'utente
result

state = {"messages": [HumanMessage("Hi, what is 2+2?")]}
result = agent_node(state)  #in questo caso, il modello dovrebbe decidere di chiamare il tool calculate, passandogli "2+2" come argomento, poi la risposta del tool (4) viene aggiunta alla history, e poi il modello risponde alla fine con la risposta finale all'utente (che dovrebbe essere 4)
result

In [ ]:
lst = [1,2,3,4]
lst[-1]  #-1 prende l'ultimo elemento della lista, quindi è il '4'

def should_continue(state: AgentState):
    last = state['messages'][-1]  #prende l'ultimo messaggio (quindi l'ultima risposta del modello), che è AImessage
    if hasattr(last, 'tool_calls') and last.tool_calls:  #CHECK SE ESISTE E SE E' TRUE IL tool_calls
        return "tools"
    else:
        return END
    #serve solo x edge routing dinamico, per decidere che nodo lanciare dopo

In [ ]:
def create_agent():
    builder = StateGraph(AgentState)
    #crete nodes
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(all_tools))  #nodo 'tools'
    #create edges
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, ["tools", END]) #ROUTING DYNAMICO! dopo nodo agent, se should_continue ritorna "tools" allora vai a nodo 'tools', altrimenti vai a END
    builder.add_edge("tools", "agent")
    graph = builder.compile()
    return graph

agent = create_agent()  #assign a var
agent

In [ ]:

query = "What is the current weather in Mumbai?"
result = agent.invoke({'messages': [HumanMessage(query)]})
result
result['messages'][-1].pretty_print()

In [ ]:
query = "What is the current weather in Mumbai? and What is 4*56 and 3-90"
result = agent.invoke({'messages': [HumanMessage(query)]})
result
result['messages'][-1].pretty_print()  #prendi l'ultimo messaggio (quindi risposta finale del modello) e stampalo in modo leggibile(cioe non in formato json)